# Phase 2 — Deep-Learning Model Zoo + Factor Research

1. Build an Alpha158 dataset.
2. Train a battery of torch models — LightGBM + LSTM + BidirectionalLSTM + TwoPathLSTM + AttentionAll + ARIMAXGBStack + AutoEncoderDNNStack.
3. Compare them via `SigAnaRecord` / IC.
4. Assemble an ML4T-style factor panel (`aqp.data.factor_lib`) and run a full Alphalens-style tearsheet.

Deps: `pip install "agentic-quant-platform[ml,ml-torch,ml-forecast]"`.

In [ ]:
from aqp.data.duckdb_engine import DuckDBHistoryProvider
from aqp.ml.features.alpha158 import Alpha158
from aqp.ml.handler import DataHandlerLP
from aqp.ml.processors import CSRankNorm, CSZScoreNorm, DropnaLabel, Fillna, PreprocessingSpec

provider = DuckDBHistoryProvider()
tickers = ["AAPL", "MSFT", "NVDA", "SPY", "QQQ", "TSLA", "GOOG", "AMZN", "META", "JPM"]
bars = provider.get_bars(tickers, start="2020-01-01", end="2024-12-31", interval="1d")
bars.head()

In [ ]:
feature_frame = Alpha158().build(bars)
print("feature shape:", feature_frame.shape)
feature_frame.head()

## Torch model zoo bake-off

In [ ]:
import contextlib
from aqp.ml.dataset import DatasetH
from aqp.ml.models.torch import (
    LSTMModel,
    BidirectionalLSTMModel,
    TwoPathLSTMModel,
    AttentionAllModel,
)
from aqp.ml.models.tree import LGBModel
from aqp.ml.models.ensemble import ARIMAXGBStack, AutoEncoderDNNStack, DEnsembleModel

segments = {
    "train": ("2020-01-01", "2023-06-30"),
    "valid": ("2023-07-01", "2023-12-31"),
    "test":  ("2024-01-01", "2024-12-31"),
}
dataset = DatasetH(handler=None, segments=segments)  # Replace None with a fitted DataHandler in your own run

candidates = {
    "LGBM":          LGBModel(n_estimators=400, learning_rate=0.05),
    "LSTM":          LSTMModel(hidden_size=64, n_epochs=5),
    "BidirLSTM":     BidirectionalLSTMModel(hidden_size=64, n_epochs=5),
    "TwoPathLSTM":   TwoPathLSTMModel(hidden_size=64, n_epochs=5),
    "AttentionAll":  AttentionAllModel(d_model=64, n_epochs=5),
    "DEnsemble":     DEnsembleModel(n_models=4),
    "ARIMAXGBStack": ARIMAXGBStack(),
    "AutoEnDNNStack":AutoEncoderDNNStack(epochs=5),
}

# Placeholder — when you've built `dataset`, uncomment:
# results = {}
# for name, m in candidates.items():
#     with contextlib.suppress(Exception):
#         m.fit(dataset)
#         results[name] = m.predict(dataset, 'test')
print(list(candidates))

## PreprocessingSpec — travels with the model

Attach a fitted processor chain to any trained model so inference servers can replay it.

In [ ]:
processors = [Fillna(fill_value=0.0), CSZScoreNorm(), DropnaLabel()]
spec = PreprocessingSpec.from_processors(
    processors=processors,
    feature_columns=[f'f{i}' for i in range(158)],
    label_column='label_5d',
    handler_cfg={
        'class': 'DataHandlerLP',
        'module_path': 'aqp.ml.handler',
        'kwargs': {},
    },
    metadata={'dataset_hash': 'demo-abc123', 'fit_window': '2020..2023'},
)
spec.summary()

## Factor research — ML4T-style

Build a factor panel and run an Alphalens-style tearsheet for each factor.

In [ ]:
from aqp.data.factor_lib import build_factor_panel
from aqp.ml.factor_eval import factor_tearsheet

factors = build_factor_panel(bars, factors=('momentum', 'rvol', 'rsi', 'bollinger_width'))
close_panel = bars.pivot(index='timestamp', columns='vt_symbol', values='close').sort_index().ffill()

for name in factors.columns:
    sheet = factor_tearsheet(name, factors[name], close_panel)
    print(f"\n=== {name} ===")
    print(sheet['ic'].round(4))
    print('turnover:', round(sheet['turnover_mean'], 4))